# v2 正式训练：Oracle-Shared（固定 $\rho_{ij}=0$）

本 notebook 仅训练 `shared_oracle_rho0`，固定使用 `learning_rate=1e-3` 和 `weight_decay=1e-4`，并使用五个 seed、16,000/2,000/4,000 数据规模、$N=256$ 训练和 $N=256/512/1024$ 零样本推理。每个 seed 完整训练 500 epoch，每 10 epoch 打印指标并保存当前 checkpoint，正式推理使用验证相对 $L_2$ 最优参数。

本模型沿用 Oracle-Shared 的四通道输入与固定门控序列，但所有 $\rho_{ij}^{\ell}=0$，因此 $a_{ij}^{\ell}=b_{ij}^{\ell}=0.5$；这些量不是可学习参数，参数树中不得出现 `rho_eta`。


## 1. 选择独占 GPU

请为各训练 notebook 指定不同的物理 GPU。本单元必须最先运行；它不会解析可能为 `[N/A]` 的利用率或显存利用字段。物理 GPU 6 被永久排除。


In [ ]:
import os
import subprocess
import sys

PHYSICAL_GPU_INDEX = 3  # 可修改，但严禁设为 6
if PHYSICAL_GPU_INDEX == 6:
    raise RuntimeError("物理 GPU 6 已永久排除。")
if "jax" in sys.modules:
    raise RuntimeError("JAX 已被导入；请重启 kernel 后从本单元开始执行。")

query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,uuid,name",
        "--format=csv,noheader,nounits",
    ],
    check=True,
    capture_output=True,
    text=True,
)
gpu_records = {}
for raw_line in query.stdout.splitlines():
    fields = [field.strip() for field in raw_line.split(",", 2)]
    if len(fields) != 3:
        raise RuntimeError(f"无法解析 nvidia-smi 行：{raw_line!r}")
    index, uuid, name = fields
    gpu_records[int(index)] = {"uuid": uuid, "name": name}
if PHYSICAL_GPU_INDEX not in gpu_records:
    raise RuntimeError(
        f"物理 GPU {PHYSICAL_GPU_INDEX} 不存在；可用编号：{sorted(gpu_records)}"
    )

os.environ["CUDA_VISIBLE_DEVICES"] = str(PHYSICAL_GPU_INDEX)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
print("已选择物理 GPU：", PHYSICAL_GPU_INDEX, gpu_records[PHYSICAL_GPU_INDEX])
print("JAX 导入后该设备会映射为逻辑 GPU 0。")


## 2. 环境、协议与固定超参数

本节验证当前 kernel 只看到一张 GPU，并创建或校验正式 v2 的不可变固定超参数 manifest。正式训练不依赖可选筛选 notebook。


In [ ]:
import json
import sys
from pathlib import Path

current_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (current_directory, *current_directory.parents)
    if (candidate / "deep_fno_shared_benchmark" / "experiment.py").is_file()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))
import jax
from deep_fno_shared_benchmark.formal_v2 import (
    ensure_fixed_hyperparameter_manifest,
    find_repository_root,
    run_formal_model,
)

MODEL = "shared_oracle_rho0"
if jax.default_backend() != "gpu" or len(jax.devices()) != 1:
    raise RuntimeError(
        f"要求恰好一张可见 GPU；backend={jax.default_backend()}, devices={jax.devices()}"
    )
repository_root = find_repository_root(repository_root)
hyperparameter_manifest, hyperparameter_manifest_path = (
    ensure_fixed_hyperparameter_manifest(repository_root)
)
print("backend:", jax.default_backend())
print("logical devices:", jax.devices())
print("fixed formal hyperparameters:")
print(json.dumps(hyperparameter_manifest["selected_hyperparameters"], indent=2))
print("immutable manifest:", hyperparameter_manifest_path)


## 3. 依次训练五个 seed

已完整通过产物审计的 seed 会被跳过；存在不完整目录时会停止并要求人工检查，防止覆盖可恢复的训练结果。运行期间进度条逐 epoch 更新，详细指标与 checkpoint 每 10 epoch 写出。


In [ ]:
result_root = run_formal_model(
    MODEL,
    repository_root=repository_root,
    seeds=(0, 1, 2, 3, 4),
    epochs=500,
    progress=True,
)
print("正式训练与产物审计完成：", result_root)


## 4. 汇总检查

`summary_all_seeds.csv` 应包含五个 seed 与三个分辨率共 15 行。每个 seed 的 `history.csv` 必须恰好包含 500 行，并同时存在周期、最佳和最终 checkpoint。


In [ ]:
import csv

summary_path = result_root / "summary_all_seeds.csv"
with summary_path.open(newline="", encoding="utf-8") as handle:
    summary_rows = list(csv.DictReader(handle))
if len(summary_rows) != 15:
    raise RuntimeError(f"汇总行数应为 15，实际为 {len(summary_rows)}")
for seed in range(5):
    run = result_root / f"seed_{seed}_run" / f"seed_{seed}" / MODEL
    with (run / "history.csv").open(newline="", encoding="utf-8") as handle:
        history = list(csv.DictReader(handle))
    if len(history) != 500:
        raise RuntimeError(f"seed={seed} history 行数不是 500")
    for filename in ("checkpoint_best.npz", "checkpoint_final.npz"):
        path = run / filename
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f"checkpoint 缺失：{path}")
print("PASS：五个 seed、三种分辨率和 checkpoint 语义均已通过静态产物检查。")
